# Master pipeline

The notebook is intentionally thin. All reusable logic lives in tested Python modules. Paid calls and training stages remain controlled by TOML switches. Long-running subprocesses stream `tqdm.auto` progress bars into the notebook output.

In [1]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from us10y_fomc.config import ProjectPaths, load_project_config
config = load_project_config(ROOT)
paths = ProjectPaths(ROOT).ensure()
config.runtime

RuntimeConfig(download_fomc_minutes=True, run_openai_preflight=True, confirm_api_key=True, run_paid_extraction=True, confirm_paid_extraction=True, retry_quarantined_pairs=False, confirm_quarantine_retry=False, max_new_pairs=3, train_price_benchmark=False, train_historical_backbone=True, train_fusion_models=True, run_walk_forward=True)

## Optional official-data refresh
This is a network step, but it has no OpenAI inference cost.

In [2]:
if config.runtime.download_fomc_minutes:
    subprocess.run([sys.executable, str(ROOT/'scripts'/'download_data.py')], cwd=ROOT, check=True)
else:
    print('Using local official data.')

Policy-rate audit: 100%|██████████| 268/268 [00:01<00:00, 265.53meeting/s]


{'treasury_rows': 11250, 'documents': 268, 'excluded': 1, 'rate_parser_disagreements': 16}


## Luna extraction
Run only after the preflight notebook. Existing qualified cache records are reused. To promote the validated pilot, run the CLI once with `--pilot /path/to/luna_paid_pilot.jsonl`.

In [3]:
PILOT_JSONL = None  # Example: Path('/absolute/path/to/luna_paid_pilot.jsonl')
command = [sys.executable, str(ROOT/'scripts'/'run_luna_extraction.py')]
if PILOT_JSONL is not None:
    command += ['--pilot', str(PILOT_JSONL)]
subprocess.run(command, cwd=ROOT, check=True)

Luna minutes extraction:   1%|          | 3/267 [01:18<1:55:48, 26.32s/pair]             


{'ready': False, 'accepted': 37, 'required': 267, 'missing': 230, 'spent_usd': 1.8217875999999986}


CompletedProcess(args=['/opt/anaconda3/envs/us10y/bin/python', '/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/scripts/run_luna_extraction.py'], returncode=0)

## Leakage-safe historical backbone

In [4]:
subprocess.run([sys.executable, str(ROOT/'scripts'/'train_historical_backbone.py')], cwd=ROOT, check=True)

Historical backbone seeds:   0%|          | 0/3 [00:00<?, ?seed/s, seed=1/3]/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/src/us10y_fomc/models/price_cnn_transformer.py:123: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(

Historical backbone seeds:  33%|███▎      | 1/3 [01:12<02:24, 72.37s/seed, seed=2/3]/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/src/us10y_fomc/models/price_cnn_transformer.py:123: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(

Historical backbone seeds:  67%|██████▋   | 2/3 [01:48<00:51, 51.07s/seed, seed=3/3]/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/src/us10y_fomc/models/price_cnn_transformer.py:123: UserWarning: enable_nested_tensor is True, but se

{'seed': 2, 'validation_balanced_accuracy': 0.4146009793339797, 'price_config': {'start_date': '1981-09-01', 'lookback': 60, 'horizon': 5, 'norm_window': 1250, 'norm_min_periods': 250, 'vol_window': 20, 'vol_scaled_target': True, 'flat_threshold_sigma': 0.25, 'flat_threshold_bp': 3.0, 'fractions': (0.65, 0.15, 0.1), 'pool_mode': 'flatten', 'pool_heads': 8, 'd_model': 128, 'cnn_channels': 64, 'n_heads': 4, 'transformer_layers': 3, 'feedforward_dim': 384, 'dropout': 0.05, 'batch_size': 128, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'warmup_steps': 500, 'max_epochs': 300, 'patience': 30, 'gradient_clip': 1.0, 'n_seeds': 3, 'quantiles': (0.05, 0.5, 0.95), 'regression_loss_weight': 0.5, 'conformal_alpha': 0.1}, 'purpose': 'fomc_historical_backbone', 'first_fomc_meeting_date': '1993-03-23T00:00:00', 'last_training_label_date': '1993-03-22T00:00:00', 'split_sizes': {'train': 1695, 'validation': 327, 'calibration': 197, 'test': 197}}


CompletedProcess(args=['/opt/anaconda3/envs/us10y/bin/python', '/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/scripts/train_historical_backbone.py'], returncode=0)

## Four-model walk-forward study
Fold, model and epoch progress is shown below. `outputs/runtime/walk_forward.jsonl` remains the durable machine-readable log.

In [5]:
subprocess.run([sys.executable, str(ROOT/'scripts'/'run_walk_forward.py')], cwd=ROOT, check=True)

/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/src/us10y_fomc/models/price_cnn_transformer.py:123: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
Fold 1/8 models:   0%|          | 0/4 [00:00<?, ?model/s, model=price_only]

fold_0_price_only:   0%|          | 0/250 [00:00<?, ?epoch/s]

fold_0_price_only:   0%|          | 0/250 [00:01<?, ?epoch/s, bacc=0.556, best=1.6069, stale=0, train=1.4419, validation=1.6069]

fold_0_price_only:   0%|          | 1/250 [00:01<05:57,  1.43s/epoch, bacc=0.556, best=1.6069, stale=0, train=1.4419, validation=1.6069]

fold_0_price_only:   0%|          | 1/250 [00:01<05:57,  1.43s/epoch, bacc=0.556, best=1.5780, stale=0, train=1.4563, validation=1.5780]

fold_0_price_only:   0%|          | 1/250 [00:01<05:57,  1.43s/epoch, bacc=0.556, best=1.5371, stale=0, train=1.4971, validation=1.5371]

fold_0_price_only:   0%|  

Saved 183 out-of-sample predictions across 8 folds.


CompletedProcess(args=['/opt/anaconda3/envs/us10y/bin/python', '/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/scripts/run_walk_forward.py'], returncode=0)

In [6]:
subprocess.run([sys.executable, str(ROOT/'scripts'/'build_report.py')], cwd=ROOT, check=True)

{'n_oos': 183, 'pooled_interval_coverage': 0.9672131147540983, 'pooled_mean_interval_width_bp': 79.11608003278688, 'models': {'fusion': {'mae_bp': 12.29382678566448, 'rmse_bp': 15.870296852902394, 'balanced_accuracy': 0.33139274518584866, 'macro_f1': 0.3162911973553601}, 'price_only': {'mae_bp': 12.388109374081965, 'rmse_bp': 15.797539436891558, 'balanced_accuracy': 0.30258247499626806, 'macro_f1': 0.277966360929735}, 'rate_only': {'mae_bp': 12.532947551491802, 'rmse_bp': 15.761679316360551, 'balanced_accuracy': 0.30974772354082697, 'macro_f1': 0.2875612610635652}, 'shuffled_text': {'mae_bp': 12.73685503262295, 'rmse_bp': 16.372766689967072, 'balanced_accuracy': 0.2984027466786087, 'macro_f1': 0.2957699787647827}}, 'paired_tests': {'fusion_vs_price_only': {'dm_statistic': -0.19523769188062548, 'p_value': 0.8454246605358987, 'mean_loss_difference': -0.09428258841748624}, 'bootstrap_fusion_vs_price_only': {'mean_difference': -0.1051945769396229, 'ci_low': -1.0337131084524511, 'ci_high': 

CompletedProcess(args=['/opt/anaconda3/envs/us10y/bin/python', '/Users/onurhalityenice/Desktop/DA/Term_Project/us10y_fomc_fusion_v7_1_v2/scripts/build_report.py'], returncode=0)